# BBBC021: 200k balanced patches for classification and Grad-CAM

Creates a five-class subset from **DMSO, Cytochalasin B, Nocodazole, Taxol, and AZ-A**. The output contains 200,000 balanced 224×224 three-channel patches in WebDataset TAR shards.

The biological hierarchy is `compound → concentration → plate → well → FOV → patch`. Splits are assigned by **plate+well before patch extraction**. Patch count is therefore not interpreted as the number of independent biological observations.

In [ ]:
%pip install -q pandas numpy pillow requests tqdm scikit-learn matplotlib

In [ ]:
from __future__ import annotations
import hashlib, io, json, math, shutil, tarfile, zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from urllib.parse import urljoin
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from PIL import Image
from tqdm.auto import tqdm

In [ ]:
@dataclass(frozen=True)
class Config:
    base_url: str = "https://data.broadinstitute.org/bbbc/BBBC021/"
    work_dir: Path = Path("/content/bbbc021_5class")
    patch_size: int = 224
    total_patches: int = 200_000
    shard_size: int = 2_000
    seed: int = 42
    train_fraction: float = 0.70
    validation_fraction: float = 0.15
    lower_percentile: float = 0.5
    upper_percentile: float = 99.8
    minimum_foreground_fraction: float = 0.02

CFG=Config()
TARGETS={
    "DMSO":"control",
    "cytochalasin B":"actin_disruption",
    "nocodazole":"microtubule_destabilization",
    "taxol":"microtubule_stabilization",
    "AZ-A":"aurora_kinase_inhibition",
}
ARCHIVES=CFG.work_dir/"archives"; IMAGES=CFG.work_dir/"images"; OUTPUT=CFG.work_dir/"patches_wds"
for p in (CFG.work_dir,ARCHIVES,IMAGES,OUTPUT): p.mkdir(parents=True,exist_ok=True)
print(asdict(CFG))

## Metadata and source selection

BBBC021 is a fixed 24-hour endpoint assay, not a time series. One metadata row corresponds to one FOV with DAPI, actin and tubulin TIFF files.

In [ ]:
def download(url: str, dst: Path, chunk_size: int=2**20) -> Path:
    if dst.exists() and dst.stat().st_size>0: return dst
    tmp=dst.with_suffix(dst.suffix+".part")
    headers={}; mode="wb"
    if tmp.exists():
        headers["Range"]=f"bytes={tmp.stat().st_size}-"; mode="ab"
    with requests.get(url,stream=True,timeout=120,headers=headers) as r:
        if r.status_code==200 and mode=="ab": tmp.unlink(); mode="wb"
        r.raise_for_status()
        with tmp.open(mode) as f:
            for chunk in r.iter_content(chunk_size):
                if chunk: f.write(chunk)
    tmp.replace(dst); return dst

meta_path=download(urljoin(CFG.base_url,"BBBC021_v1_image.csv"),CFG.work_dir/"BBBC021_v1_image.csv")
moa_path=download(urljoin(CFG.base_url,"BBBC021_v1_moa.csv"),CFG.work_dir/"BBBC021_v1_moa.csv")
meta=pd.read_csv(meta_path); moa=pd.read_csv(moa_path)
COMPOUND="Image_Metadata_Compound"; CONC="Image_Metadata_Concentration"; PLATE="Image_Metadata_Plate_DAPI"; WELL="Image_Metadata_Well_DAPI"
CHANNELS={"dapi":("Image_PathName_DAPI","Image_FileName_DAPI"),"actin":("Image_PathName_Actin","Image_FileName_Actin"),"tubulin":("Image_PathName_Tubulin","Image_FileName_Tubulin")}
available={str(x).casefold():str(x) for x in meta[COMPOUND].dropna().unique()}
missing=[x for x in TARGETS if x.casefold() not in available]
if missing: raise ValueError(f"Missing compounds: {missing}")
selected=[available[x.casefold()] for x in TARGETS]
subset=meta[meta[COMPOUND].isin(selected)].copy()
subset["class_name"]=subset[COMPOUND].map(TARGETS)
subset["group_id"]=subset[PLATE].astype(str)+"::"+subset[WELL].astype(str)
display(subset.groupby(["class_name",COMPOUND]).agg(fovs=("ImageNumber","size"),wells=("group_id","nunique"),plates=(PLATE,"nunique"),concentrations=(CONC,"nunique")).reset_index())

In [ ]:
def assign_splits(df: pd.DataFrame) -> pd.DataFrame:
    out=[]
    for cls,g in df.groupby("class_name",sort=True):
        g=g.copy(); groups=g["group_id"].drop_duplicates().to_numpy()
        if len(groups)<3: raise ValueError(f"{cls} has only {len(groups)} wells")
        rng=np.random.default_rng(CFG.seed); rng.shuffle(groups)
        n=len(groups); nt=max(1,round(CFG.train_fraction*n)); nv=max(1,round(CFG.validation_fraction*n))
        if nt+nv>=n: nt=n-2; nv=1
        mapping={x:"train" for x in groups[:nt]}
        mapping.update({x:"validation" for x in groups[nt:nt+nv]})
        mapping.update({x:"test" for x in groups[nt+nv:]})
        g["split"]=g["group_id"].map(mapping); out.append(g)
    return pd.concat(out,ignore_index=True)
subset=assign_splits(subset)
display(subset.groupby(["class_name","split"]).agg(fovs=("ImageNumber","size"),wells=("group_id","nunique"),plates=(PLATE,"nunique")).reset_index())

## Download only required plate archives

Archives are distributed per plate, so the source download can be much larger than the final five-class subset.

In [ ]:
def archive_name(plate: str) -> str: return f"BBBC021_v1_images_{plate}.zip"
def remote_size(url: str):
    try:
        r=requests.head(url,allow_redirects=True,timeout=30); r.raise_for_status(); v=r.headers.get("Content-Length"); return int(v) if v else None
    except requests.RequestException: return None
plates=sorted(subset[PLATE].astype(str).unique())
manifest=pd.DataFrame({"plate":plates})
manifest["archive"]=manifest["plate"].map(archive_name)
manifest["url"]=manifest["archive"].map(lambda x:urljoin(CFG.base_url,x))
manifest["bytes"]=[remote_size(x) for x in tqdm(manifest["url"],desc="HEAD")]
manifest["GiB"]=manifest["bytes"]/2**30
display(manifest); print(f"Required source download: {manifest['GiB'].sum():.2f} GiB")
for row in tqdm(manifest.itertuples(index=False),total=len(manifest),desc="Downloading"):
    download(row.url,ARCHIVES/row.archive)

In [ ]:
needed={str(v) for _,fcol in CHANNELS.values() for v in subset[fcol].dropna()}
found=set()
for zpath in tqdm(sorted(ARCHIVES.glob("*.zip")),desc="Extracting selected TIFFs"):
    with zipfile.ZipFile(zpath) as z:
        matches={Path(m.filename).name:m for m in z.infolist() if Path(m.filename).name in needed}
        for name,m in matches.items():
            dst=IMAGES/name
            if not dst.exists():
                with z.open(m) as src, dst.open("wb") as out: shutil.copyfileobj(src,out)
            found.add(name)
missing=needed-found
if missing: raise FileNotFoundError(f"Missing {len(missing)} TIFFs, examples: {sorted(missing)[:10]}")
print(f"Extracted {len(found):,} channel TIFFs")

## Three-channel loading and pilot size estimate

Channels are ordered DAPI, actin, tubulin. Each channel is percentile-scaled to uint8 before lossless PNG encoding.

In [ ]:
def norm_channel(x: np.ndarray) -> np.ndarray:
    x=x.astype(np.float32); lo=np.percentile(x,CFG.lower_percentile); hi=np.percentile(x,CFG.upper_percentile)
    if hi<=lo: return np.zeros_like(x,dtype=np.uint8)
    return np.round(255*np.clip((x-lo)/(hi-lo),0,1)).astype(np.uint8)
def load_fov(row: pd.Series) -> np.ndarray:
    arr=[]
    for name in ("dapi","actin","tubulin"):
        _,fcol=CHANNELS[name]
        with Image.open(IMAGES/str(row[fcol])) as im: arr.append(norm_channel(np.asarray(im)))
    if len({a.shape for a in arr})!=1: raise ValueError("Channel shape mismatch")
    return np.stack(arr,axis=-1)
def encode_png(x: np.ndarray) -> bytes:
    b=io.BytesIO(); Image.fromarray(x).save(b,format="PNG"); return b.getvalue()
def random_patch(img: np.ndarray,rng: np.random.Generator):
    h,w,_=img.shape; p=CFG.patch_size
    if h<p or w<p: raise ValueError(f"FOV {img.shape} smaller than patch")
    y=int(rng.integers(0,h-p+1)); x=int(rng.integers(0,w-p+1)); return img[y:y+p,x:x+p],y,x
def foreground_fraction(patch): return float((np.max(patch,axis=-1)>=8).mean())
rng=np.random.default_rng(CFG.seed); pilot=[]
for _,row in tqdm(subset.sample(min(100,len(subset)),random_state=CFG.seed).iterrows(),total=min(100,len(subset))):
    patch,_,_=random_patch(load_fov(row),rng); pilot.append(len(encode_png(patch)))
raw8=CFG.total_patches*CFG.patch_size**2*3/2**30
est=np.mean(pilot)*CFG.total_patches/2**30
display(pd.DataFrame({"representation":["PNG pilot estimate","raw uint8","raw uint16","raw float32"],"GiB":[est,raw8,2*raw8,4*raw8]}))
print(f"Mean pilot PNG: {np.mean(pilot)/1024:.1f} KiB")

## Generate 200,000 balanced patches

Each class receives 40,000 patches. Exact duplicate coordinates within an FOV are rejected. Overlapping patches remain correlated, especially for classes with few source FOVs.

In [ ]:
def counts(total, fractions):
    raw={k:total*v for k,v in fractions.items()}; out={k:math.floor(v) for k,v in raw.items()}
    for k in sorted(raw,key=lambda x:raw[x]-out[x],reverse=True)[:total-sum(out.values())]: out[k]+=1
    return out
per_class=CFG.total_patches//len(TARGETS)
if per_class*len(TARGETS)!=CFG.total_patches: raise ValueError("total_patches must divide by classes")
per_split=counts(per_class,{"train":CFG.train_fraction,"validation":CFG.validation_fraction,"test":1-CFG.train_fraction-CFG.validation_fraction})
allocation=pd.DataFrame([{"class_name":c,"split":s,"patches":n} for c in sorted(TARGETS.values()) for s,n in per_split.items()])
display(allocation)

In [ ]:
class ShardWriter:
    def __init__(self,out:Path,shard_size:int): self.out=out; self.shard_size=shard_size; self.n=0; self.i=-1; self.tar=None
    def __enter__(self): return self
    def __exit__(self,*args): self.close()
    def _next(self):
        if self.tar: self.tar.close()
        self.i+=1; self.tar=tarfile.open(self.out/f"bbbc021-{self.i:05d}.tar","w")
    def write(self,key,png,meta):
        if self.n%self.shard_size==0: self._next()
        for name,payload in {f"{key}.png":png,f"{key}.json":json.dumps(meta,sort_keys=True).encode()}.items():
            info=tarfile.TarInfo(name); info.size=len(payload); self.tar.addfile(info,io.BytesIO(payload))
        self.n+=1
    def close(self):
        if self.tar: self.tar.close(); self.tar=None

def row_id(row):
    s=f"{row[PLATE]}|{row[WELL]}|{row['ImageNumber']}"; return hashlib.sha1(s.encode()).hexdigest()[:12]
def generate(group,count,cls,split,writer,records,rng):
    rows=[r for _,r in group.iterrows()]
    if not rows: raise ValueError(f"No FOVs for {cls}/{split}")
    cache={}; used={}; accepted=attempts=0
    bar=tqdm(total=count,desc=f"{cls}/{split}")
    while accepted<count and attempts<20*count:
        row=rows[accepted%len(rows)]; rid=row_id(row)
        if rid not in cache: cache[rid]=load_fov(row); used[rid]=set()
        patch,y,x=random_patch(cache[rid],rng); attempts+=1
        if (y,x) in used[rid]: continue
        used[rid].add((y,x))
        if foreground_fraction(patch)<CFG.minimum_foreground_fraction: continue
        key=f"{writer.n:09d}"
        m={"key":key,"class_name":cls,"split":split,"compound":str(row[COMPOUND]),"concentration":float(row[CONC]),"plate":str(row[PLATE]),"well":str(row[WELL]),"fov_image_number":int(row['ImageNumber']),"source_fov_id":rid,"patch_y":y,"patch_x":x,"patch_size":CFG.patch_size,"channel_order":["DAPI","Actin","Tubulin"]}
        writer.write(key,encode_png(patch),m); records.append(m); accepted+=1; bar.update(1)
        if len(cache)>16: cache.pop(next(iter(cache)))
    bar.close()
    if accepted<count: raise RuntimeError(f"Only generated {accepted}/{count} for {cls}/{split}")

for p in OUTPUT.glob("bbbc021-*.tar"): p.unlink()
records=[]; rng=np.random.default_rng(CFG.seed)
with ShardWriter(OUTPUT,CFG.shard_size) as writer:
    for cls in sorted(TARGETS.values()):
        for split in ("train","validation","test"):
            g=subset[(subset.class_name==cls)&(subset.split==split)]
            n=int(allocation[(allocation.class_name==cls)&(allocation.split==split)].patches.iloc[0])
            generate(g,n,cls,split,writer,records,rng)
patch_manifest=pd.DataFrame(records)
patch_manifest.to_csv(OUTPUT/"manifest.csv",index=False)
(OUTPUT/"config.json").write_text(json.dumps(asdict(CFG),indent=2,default=str))
print(f"Wrote {len(patch_manifest):,} patches")

## Validation and effective sample structure

In [ ]:
assert len(patch_manifest)==CFG.total_patches
assert patch_manifest.groupby("class_name").size().nunique()==1
assert patch_manifest.groupby(["plate","well"])["split"].nunique().max()==1
assert patch_manifest.groupby("source_fov_id")["split"].nunique().max()==1
summary=patch_manifest.groupby(["class_name","split"]).agg(patches=("key","size"),source_fovs=("source_fov_id","nunique"),wells=("well","nunique"),plates=("plate","nunique"),patches_per_fov=("source_fov_id",lambda x:len(x)/x.nunique())).reset_index()
display(summary)
final_bytes=sum(p.stat().st_size for p in OUTPUT.glob("*") if p.is_file())
print(f"Final dataset: {final_bytes/2**30:.2f} GiB in {len(list(OUTPUT.glob('*.tar')))} shards")
ax=patch_manifest.groupby(["class_name","split"]).size().unstack(fill_value=0).plot(kind="bar",stacked=True,figsize=(10,5))
ax.set_ylabel("Patches"); ax.set_title("Patch allocation"); plt.xticks(rotation=35,ha="right"); plt.tight_layout(); plt.show()

## Optional Google Drive export

TAR shards are used because copying 200,000 individual files to Drive is inefficient.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# destination=Path('/content/drive/MyDrive/datasets/bbbc021_5class_200k')
# shutil.copytree(OUTPUT,destination,dirs_exist_ok=True)
# print(destination)

## Size interpretation

For 200,000 patches of size 224×224×3, raw uint8 storage is about **28.0 GiB**, uint16 about **56.1 GiB**, and float32 about **112.2 GiB**. Lossless PNG WebDataset storage is expected roughly in the **10–25 GiB** range, but the pilot estimate produced by the notebook is more informative. The required source ZIP download is calculated separately from the exact plates selected.